# Notebook 03 — SCD Type 2: Player Price History

Track every player price change over time using a **Slowly Changing Dimension
Type 2** pattern. Each row has `effective_from` / `effective_to` date columns
and an `is_current` flag, so you can query the latest price or reconstruct the
full price timeline for any player.

**First run**: seeds the table with every player's current price.
**Subsequent runs**: detects price changes, closes old records, and inserts new ones.

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
SCD_TABLE = "fpl_project.silver.player_price_history"

# Pull the latest player prices from the silver layer
incoming = spark.table("fpl_project.silver.players") \
    .select("player_id", "display_name", "current_price", "team_id")

if not spark.catalog.tableExists(SCD_TABLE):
    # --- First run: initialize every player with today's price ---
    scd_init = incoming \
        .withColumn("effective_from", F.current_date()) \
        .withColumn("effective_to", F.lit("9999-12-31").cast("date")) \
        .withColumn("is_current", F.lit(True))

    scd_init.write.format("delta").saveAsTable(SCD_TABLE)
    print("SCD table initialized.")
else:
    # --- Incremental run: detect and record price changes ---
    current_records = spark.table(SCD_TABLE).filter("is_current = true")

    # Join incoming prices against current SCD records to find differences
    changes = incoming.alias("s").join(
        current_records.alias("t"),
        "player_id"
    ).filter("s.current_price != t.current_price")

    change_count = changes.count()

    if change_count > 0:
        target = DeltaTable.forName(spark, SCD_TABLE)

        # Step 1: Close out old records — set their end date to today
        # and mark them as no longer current
        target.alias("t").merge(
            changes.select(F.col("s.player_id").alias("player_id")).alias("c"),
            "t.player_id = c.player_id AND t.is_current = true"
        ).whenMatchedUpdate(set={
            "effective_to": "current_date()",
            "is_current": "false"
        }).execute()

        # Step 2: Insert new current records with today's effective_from
        new_records = changes.select(
            F.col("s.player_id"),
            F.col("s.display_name"),
            F.col("s.current_price"),
            F.col("s.team_id"),
            F.current_date().alias("effective_from"),
            F.lit("9999-12-31").cast("date").alias("effective_to"),
            F.lit(True).alias("is_current"),
        )

        new_records.write.format("delta").mode("append") \
            .saveAsTable(SCD_TABLE)

        print(f"SCD updated: {change_count} price changes recorded.")
    else:
        print("No price changes detected.")